# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [1]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 HTTPS 저장소 URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")

    # 사용자가 복사한 값에 `git clone`이 앞에 붙어 있어도 URL 부분만 사용합니다.
    if url.startswith("git clone "):
        url = url[len("git clone "):].strip()

    # github.com/USER/REPO, http://github.com/USER/REPO, https://github.com/USER/REPO를 모두 허용합니다.
    if url.startswith("github.com/"):
        url = "https://" + url
    elif url.startswith("http://github.com/"):
        url = "https://" + url[len("http://"):]

    if not url.startswith("https://github.com/"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")

    # 브라우저 주소창에서 /tree/브랜치 또는 /blob/브랜치 URL을 복사해도 repo URL만 남깁니다.
    for marker in ("/tree/", "/blob/"):
        if marker in url:
            url = url.split(marker, 1)[0]

    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


def get_repo_name(repo_url: str) -> str:
    """https://github.com/USER/REPO.git에서 REPO 이름만 꺼냅니다."""
    return repo_url.removesuffix(".git").rstrip("/").split("/")[-1]


def checkout_branch(repo_dir: Path, branch: str) -> None:
    """이미 clone된 저장소에서 사용자가 고른 브랜치로 전환합니다."""
    subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)

    # 로컬에 브랜치가 있으면 바로 checkout하고, 없으면 origin/<branch>에서 새로 만듭니다.
    result = subprocess.run(["git", "checkout", branch], cwd=repo_dir)
    if result.returncode != 0:
        subprocess.run(["git", "checkout", "-b", branch, f"origin/{branch}"], cwd=repo_dir, check=True)


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_input = input("GitHub 저장소 URL (예: https://github.com/USERNAME/gpt-lab.git): ")
    repo_url = normalize_github_url(repo_input)
    branch = input("브랜치명 (예: main, dev/stage, 비우면 기본 브랜치): ").strip()
    branch = branch or None

    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = get_repo_name(repo_url)
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        clone_cmd = ["git", "clone"]
        if branch is not None:
            clone_cmd += ["--branch", branch, "--single-branch"]
        clone_cmd += [clone_url, str(repo_dir)]
        subprocess.run(clone_cmd, check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
        if branch is not None:
            checkout_branch(repo_dir, branch)

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")


Repo: /Users/juhoseok/Desktop/week14-team-03-gpt-lab


In [2]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [3]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

이미 존재합니다: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/ratings_train.txt
이미 존재합니다: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/ratings_test.txt
사전 학습 train 텍스트: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/nsmc_lm_train.txt (1,379,486자)
사전 학습 val 텍스트: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/nsmc_lm_val.txt (120,560자)
감성 분류 train: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/nsmc_sentiment_train.jsonl (137,996개)
감성 분류 val: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/nsmc_sentiment_val.jsonl (11,999개)
감성 분류 test: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/nsmc_sentiment_test.jsonl (49,997개)
LM train exists: True /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/nsmc_lm_train.txt
LM val exists: True /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/nsmc_lm_val.txt


In [4]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

train chars: 1379486
val chars: 120560
개재미없다. 감독의 연출력의 한계
이제서야 보게된 대 명작 연출미가 정말 훌륭하다!!!!!!!!
소주미라클을 만들어라
귀여운 캐릭터들도 많이 나와서 보러 가야 겠어요..
블랙 코미디가 싫어요.
평점깎고싶다10글자
TV시리즈가 너무재밌어서 영화는 기대안하고 봤는데 역시....최고네요
개인적 공감이 글쎄?
시작은 니시지마 때문에 봤는데 나름 괜찮은 영화 봤다고


## 2.1 실험 설정

아래 셀의 `EXPERIMENT_PRESET`만 바꾸면 데이터 크기, BPE vocab size, context length, 모델 크기, 결과 저장 경로가 함께 바뀝니다.


In [ ]:
# 실험 규모를 여기서 한 번에 바꿉니다.
# - smoke: 구현 확인과 한 배치 학습 확인
# - light: 빠른 실험
# - basic: 기본 제출 규모
EXPERIMENT_PRESET = "light"  # "smoke" | "light" | "basic"

PRESETS = {
    "smoke": {
        "corpus_chars": 5_000,
        "vocab_size": 300,
        "context_length": 32,
        "batch_size": 8,
        "emb_dim": 64,
        "n_heads": 4,
        "n_layers": 2,
        "drop_rate": 0.1,
        "learning_rate": 3e-4,
        "num_epochs": 1,
        "eval_freq": 20,
        "eval_iter": 5,
    },
    "light": {
        "corpus_chars": 500_000,
        "vocab_size": 2_000,
        "context_length": 64,
        "batch_size": 8,
        "emb_dim": 128,
        "n_heads": 4,
        "n_layers": 2,
        "drop_rate": 0.1,
        "learning_rate": 3e-4,
        "num_epochs": 3,
        "eval_freq": 100,
        "eval_iter": 10,
    },
    "basic": {
        "corpus_chars": 1_500_000,
        "vocab_size": 3_000,
        "context_length": 128,
        "batch_size": 8,
        "emb_dim": 192,
        "n_heads": 4,
        "n_layers": 4,
        "drop_rate": 0.1,
        "learning_rate": 3e-4,
        "num_epochs": 3,
        "eval_freq": 200,
        "eval_iter": 20,
    },
}

CFG = PRESETS[EXPERIMENT_PRESET].copy()

MODEL_CONFIG = {
    "vocab_size": CFG["vocab_size"],
    "context_length": CFG["context_length"],
    "emb_dim": CFG["emb_dim"],
    "n_heads": CFG["n_heads"],
    "n_layers": CFG["n_layers"],
    "drop_rate": CFG["drop_rate"],
    "qkv_bias": False,
}

BPE_CORPUS = corpus[:CFG["corpus_chars"]]
VOCAB_PATH = repo_dir / "data" / f"vocab_{EXPERIMENT_PRESET}_{CFG['vocab_size']}.json"
RESULTS_DIR = repo_dir / "results"
PRETRAIN_RESULTS_PATH = RESULTS_DIR / f"pretrain_{EXPERIMENT_PRESET}.json"
SENTIMENT_RESULTS_PATH = RESULTS_DIR / f"sentiment_{EXPERIMENT_PRESET}.json"

print("preset:", EXPERIMENT_PRESET)
print("config:", CFG)
print("BPE corpus chars:", len(BPE_CORPUS))
print("vocab path:", VOCAB_PATH)
print("pretrain results path:", PRETRAIN_RESULTS_PATH)
print("sentiment results path:", SENTIMENT_RESULTS_PATH)


preset: light
config: {'corpus_chars': 500000, 'vocab_size': 2000, 'context_length': 64, 'batch_size': 8, 'emb_dim': 256, 'n_heads': 4, 'n_layers': 4, 'drop_rate': 0.1, 'learning_rate': 0.0003, 'num_epochs': 3, 'eval_freq': 100, 'eval_iter': 10}
BPE corpus chars: 500000
vocab path: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/vocab_light_2000.json
pretrain results path: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/results/pretrain_light.json
sentiment results path: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/results/sentiment_light.json


## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [6]:
run_pytest("tests/test_bpe.py")

실행 명령: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python -m pytest tests/test_bpe.py -v
============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.0.3, pluggy-1.6.0 -- /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/juhoseok/Desktop/week14-team-03-gpt-lab
collecting ... collected 8 items

tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [ 12%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 25%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 37%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 50%]
tests/test_bpe.py::TestBPETokenizer::test_decode_replace_handles_invalid_utf8_generated_bytes PASSED [ 62%]
tests/test_bpe.py::TestBPETokenizer::test_get_special_ids PASSED         [ 75%]
tests/test_bpe.py::TestBPETrain::test_train_increases

0

In [7]:
# 설정 셀의 vocab_size/corpus_chars를 사용해 vocabulary를 저장하고 재사용합니다.
try:
    from bpe import BPETokenizer

    if not BPE_CORPUS:
        raise RuntimeError("BPE 학습용 corpus가 비어 있습니다. 데이터 준비 셀을 먼저 실행하세요.")

    tokenizer = BPETokenizer(vocab_size=CFG["vocab_size"])

    if VOCAB_PATH.exists():
        tokenizer.load(VOCAB_PATH)
        print("기존 vocabulary 로드:", VOCAB_PATH)
    else:
        tokenizer.train(BPE_CORPUS)
        tokenizer.save(VOCAB_PATH)
        print("새 vocabulary 저장:", VOCAB_PATH)

    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print("actual vocab size:", len(tokenizer.id_to_token))
    print("sample ids:", ids[:20])
    print("decoded:", tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)


기존 vocabulary 로드: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/vocab_light_2000.json
actual vocab size: 2000
sample ids: [2, 267, 1162, 538, 841, 1879, 36, 73, 114, 107, 112, 109, 119, 108, 498, 54, 55, 3]
decoded: 이 영화는 정말 좋았다! English 123


## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [8]:
run_pytest("tests/test_dataset.py")

실행 명령: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python -m pytest tests/test_dataset.py -v
============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.0.3, pluggy-1.6.0 -- /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/juhoseok/Desktop/week14-team-03-gpt-lab
collecting ... collected 4 items

tests/test_dataset.py::TestGPTDataset::test_dataset_length PASSED        [ 25%]
tests/test_dataset.py::TestGPTDataset::test_dataset_getitem_shape PASSED [ 50%]
tests/test_dataset.py::TestCreateDataloader::test_dataloader_batch_shape PASSED [ 75%]
tests/test_dataset.py::TestInputEmbedding::test_input_embedding_shape PASSED [100%]

============================== 4 passed in 1.57s ===============================


선택한 테스트를 통과했습니다.


0

In [9]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    if "tokenizer" not in globals():
        tokenizer = BPETokenizer(vocab_size=CFG["vocab_size"])
        tokenizer.load(VOCAB_PATH)

    token_ids = tokenizer.encode(BPE_CORPUS)
    loader = create_dataloader(
        token_ids,
        context_length=CFG["context_length"],
        batch_size=min(2, CFG["batch_size"]),
        shuffle=False,
    )
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(
        vocab_size=CFG["vocab_size"],
        emb_dim=CFG["emb_dim"],
        context_length=CFG["context_length"],
        drop_rate=0.0,
    )
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)


torch.Size([2, 64]) torch.Size([2, 64]) torch.Size([2, 64, 256])


## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [10]:
run_pytest("tests/test_attention.py")

실행 명령: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python -m pytest tests/test_attention.py -v
============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.0.3, pluggy-1.6.0 -- /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/juhoseok/Desktop/week14-team-03-gpt-lab
collecting ... collected 2 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [ 50%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [100%]

============================== 2 passed in 1.12s ===============================


선택한 테스트를 통과했습니다.


0

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [11]:
run_pytest("tests/test_model.py")

실행 명령: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python -m pytest tests/test_model.py -v
============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.0.3, pluggy-1.6.0 -- /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/juhoseok/Desktop/week14-team-03-gpt-lab
collecting ... collected 7 items

tests/test_model.py::TestLayerNorm::test_layernorm_shape PASSED          [ 14%]
tests/test_model.py::TestGELU::test_gelu_shape PASSED                    [ 28%]
tests/test_model.py::TestFeedForward::test_feedforward_shape PASSED      [ 42%]
tests/test_model.py::TestTransformerBlock::test_transformer_block_shape PASSED [ 57%]
tests/test_model.py::TestGPTModel::test_gpt_forward_shape PASSED         [ 71%]
tests/test_model.py::TestGPTModel::test_gpt_forward_with_targets_returns_loss PASSED [ 85%]
tests/test_model.py::TestGenerateTextSimple::test_generate_text_simple_shap

0

In [12]:
try:
    import torch
    from model import GPTModel

    config = MODEL_CONFIG.copy()
    model = GPTModel(config)
    seq_len = min(16, config["context_length"])
    x = torch.randint(0, config["vocab_size"], (2, seq_len))
    logits = model(x)
    print("model config:", config)
    print("logits shape:", logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)


model config: {'vocab_size': 2000, 'context_length': 64, 'emb_dim': 256, 'n_heads': 4, 'n_layers': 4, 'drop_rate': 0.1, 'qkv_bias': False}
logits shape: torch.Size([2, 16, 2000])


## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [13]:
run_pytest("tests/test_train.py")

실행 명령: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python -m pytest tests/test_train.py -v
============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.0.3, pluggy-1.6.0 -- /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/juhoseok/Desktop/week14-team-03-gpt-lab
collecting ... collected 7 items

tests/test_train.py::TestCalcLossBatch::test_calc_loss_batch_returns_scalar PASSED [ 14%]
tests/test_train.py::TestCalcLossLoader::test_calc_loss_loader_returns_float PASSED [ 28%]
tests/test_train.py::TestCalcLossLoader::test_calc_loss_loader_restores_original_model_mode PASSED [ 42%]
tests/test_train.py::TestCheckpoint::test_save_load_checkpoint_restores_epoch_and_step PASSED [ 57%]
tests/test_train.py::TestGenerate::test_generate_shape PASSED            [ 71%]
tests/test_train.py::TestTrainModelResults::test_train_model_saves_pretraining_history PASSED [ 85%]
test

0

In [14]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    if "tokenizer" not in globals():
        tokenizer = BPETokenizer(vocab_size=CFG["vocab_size"])
        tokenizer.load(VOCAB_PATH)

    token_ids = tokenizer.encode(BPE_CORPUS)
    loader = create_dataloader(
        token_ids,
        context_length=CFG["context_length"],
        batch_size=min(2, CFG["batch_size"]),
        shuffle=False,
    )
    inp, tgt = next(iter(loader))
    config = MODEL_CONFIG.copy()
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("preset:", EXPERIMENT_PRESET)
    print("vocab path:", VOCAB_PATH)
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)


preset: light
vocab path: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/data/vocab_light_2000.json
smoke loss: 7.714392185211182


### 7.1 사전학습 실행 및 결과 저장

아래 셀을 실행해야 `results/pretrain_<preset>.json` 파일이 생성됩니다. `basic` 설정은 시간이 오래 걸릴 수 있으므로 먼저 `smoke`로 확인하세요.


In [15]:
# 사전학습을 실제로 실행하고 train/validation loss와 생성 샘플을 JSON으로 저장합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import train_model

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if "tokenizer" not in globals():
        tokenizer = BPETokenizer(vocab_size=CFG["vocab_size"])
        tokenizer.load(VOCAB_PATH)

    train_token_ids = tokenizer.encode(BPE_CORPUS)
    val_chars = max(CFG["context_length"] * 100, min(len(val_corpus), CFG["corpus_chars"] // 10))
    val_token_ids = tokenizer.encode(val_corpus[:val_chars])

    train_loader = create_dataloader(
        train_token_ids,
        context_length=CFG["context_length"],
        batch_size=CFG["batch_size"],
        stride=CFG["context_length"],
        shuffle=True,
        drop_last=True,
    )
    val_loader = create_dataloader(
        val_token_ids,
        context_length=CFG["context_length"],
        batch_size=CFG["batch_size"],
        stride=CFG["context_length"],
        shuffle=False,
        drop_last=False,
    )

    model = GPTModel(MODEL_CONFIG.copy()).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"])

    train_losses = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        device=device,
        num_epochs=CFG["num_epochs"],
        eval_freq=CFG["eval_freq"],
        eval_iter=CFG["eval_iter"],
        start_context="이 영화",
        tokenizer=tokenizer,
        results_path=PRETRAIN_RESULTS_PATH,
    )

    print("train losses:", train_losses)
    print("pretrain results saved:", PRETRAIN_RESULTS_PATH, PRETRAIN_RESULTS_PATH.exists())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)


Epoch 1, step 100: train loss 6.9735, val loss 6.9834
이 영화�을을�을을한�가은하고서나지아의.
기나한다.
의을을
은이하�에은이영화로...이가고는만을도... 영화에 은한의

Epoch 1, step 200: train loss 6.9512, val loss 6.9719
이 영화가는인 이이영화을을....지인도게이도시 이는
이 이기하고 그 영화의시�도가의들가
인지��다리한이��.도기가�
Epoch 1, step 300: train loss 6.8933, val loss 6.9351
이 영화가..�이에을도 영화지�고영화...서도를히가
우,자은의가 영화에스의아도도서의 너무기들�거다,
지이는은도의니이
Epoch 1, step 400: train loss 6.7307, val loss 6.7792
이 영화의 진는에.
지의 �어다 이기하고
거 이야.를�의가... 가는데 그이 일어이
고 나라니 그한 스토리.정치
 영화의 전어하고마가한
Epoch 1, step 500: train loss 6.4790, val loss 6.5603
이 영화 a t 뭐이 재밌다 그리서 정말이 s은 한을 말영화라가 진진하고, 쿌리스의 ㄷㄷ ㅎ���, D하고 어거리 C e
Epoch 1, step 600: train loss 6.2229, val loss 6.3244
이 영화도 잘히 잘하고도 생각한 다리아
개한다
유도 보시시서..자인을 할가 보면서.도 그대로음이 그러나고 감동는 미이 뭐을 생각한영화의 미
Epoch 2, step 700: train loss 6.0087, val loss 6.0921
이 영화로 보고있었음 ㅉㅉ...
진짜..
너무이 재밌었지만 그게 봤지만가 뭐
평점인트. 너무 지루한 소내한 소재..
재밌게 만든도 아잔하고 그래서 이런 이걸의 그걸
Epoch 2, step 800: train loss 5.7923, val loss 5.9328
이 영화...
진짜, 연기에 감동 a 6이 아닌어
기대이 다행지에 가지고자들이 안진 영화의 그러다운영화에

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [16]:
run_pytest("tests/test_finetune.py")

실행 명령: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python -m pytest tests/test_finetune.py -v
============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.0.3, pluggy-1.6.0 -- /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/juhoseok/Desktop/week14-team-03-gpt-lab
collecting ... collected 6 items

tests/test_finetune.py::TestMakeSentimentDataset::test_make_sentiment_dataset_splits_rows PASSED [ 16%]
tests/test_finetune.py::TestMakeSentimentDataset::test_make_sentiment_dataset_writes_required_jsonl_files PASSED [ 33%]
tests/test_finetune.py::TestReviewSentimentDataset::test_review_sentiment_dataset_getitem PASSED [ 50%]
tests/test_finetune.py::TestGPTForSequenceClassification::test_sequence_classification_shape PASSED [ 66%]
tests/test_finetune.py::TestSentimentTrainEval::test_train_eval_functions_exist PASSED [ 83%]
tests/test_finetune.py::TestSentimentTrai

0

### 8.1 감성 분류 실행 및 결과 저장

아래 셀을 실행해야 `results/sentiment_<preset>.json` 파일이 생성됩니다. train/validation loss와 accuracy는 매 epoch 저장하고, test loss와 accuracy는 마지막 epoch 모델로 저장합니다.


In [17]:
# 감성 분류 fine-tuning을 실행하고 train/validation/test loss와 accuracy를 JSON으로 저장합니다.
try:
    import json
    import torch
    from torch.utils.data import DataLoader
    from bpe import BPETokenizer
    from model import GPTModel
    from finetune import (
        ReviewSentimentDataset,
        GPTForSequenceClassification,
        train_epoch_sentiment,
        evaluate_sentiment,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if "tokenizer" not in globals():
        tokenizer = BPETokenizer(vocab_size=CFG["vocab_size"])
        tokenizer.load(VOCAB_PATH)

    def read_jsonl(path, limit=None):
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                rows.append(json.loads(line))
                if limit is not None and len(rows) >= limit:
                    break
        return rows

    sentiment_limits = {
        "smoke": {"train": 512, "val": 128, "test": 128},
        "light": {"train": 5_000, "val": 1_000, "test": 1_000},
        "basic": {"train": None, "val": None, "test": None},
    }[EXPERIMENT_PRESET]

    train_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_train.jsonl", sentiment_limits["train"])
    val_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_val.jsonl", sentiment_limits["val"])
    test_data = read_jsonl(repo_dir / "data" / "nsmc_sentiment_test.jsonl", sentiment_limits["test"])

    train_ds = ReviewSentimentDataset(train_data, tokenizer, max_length=CFG["context_length"])
    val_ds = ReviewSentimentDataset(val_data, tokenizer, max_length=CFG["context_length"])
    test_ds = ReviewSentimentDataset(test_data, tokenizer, max_length=CFG["context_length"])

    train_loader_cls = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True)
    val_loader_cls = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False)
    test_loader_cls = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False)

    # 위 사전학습 셀을 실행했다면 그 GPTModel을 backbone으로 사용합니다.
    # 실행하지 않았다면 같은 설정의 새 backbone으로 분류 흐름만 확인합니다.
    if "model" in globals() and isinstance(model, GPTModel):
        backbone = model
    else:
        backbone = GPTModel(MODEL_CONFIG.copy())

    clf_model = GPTForSequenceClassification(backbone, num_labels=2).to(device)
    optimizer = torch.optim.AdamW(clf_model.parameters(), lr=CFG["learning_rate"])

    if SENTIMENT_RESULTS_PATH.exists():
        SENTIMENT_RESULTS_PATH.unlink()

    for epoch in range(1, CFG["num_epochs"] + 1):
        train_loss, train_acc = train_epoch_sentiment(
            clf_model,
            train_loader_cls,
            optimizer,
            device,
            epoch=epoch,
            results_path=SENTIMENT_RESULTS_PATH,
        )
        val_loss, val_acc = evaluate_sentiment(
            clf_model,
            val_loader_cls,
            device,
            split="val",
            epoch=epoch,
            results_path=SENTIMENT_RESULTS_PATH,
        )

        print(
            f"epoch {epoch} "
            f"train loss={train_loss:.4f}, accuracy={train_acc:.4f} | "
            f"val loss={val_loss:.4f}, accuracy={val_acc:.4f}"
        )

    test_loss, test_acc = evaluate_sentiment(
        clf_model,
        test_loader_cls,
        device,
        split="test",
        epoch=CFG["num_epochs"],
        results_path=SENTIMENT_RESULTS_PATH,
    )

    print(f"test loss={test_loss:.4f}, accuracy={test_acc:.4f}")
    print("sentiment results saved:", SENTIMENT_RESULTS_PATH, SENTIMENT_RESULTS_PATH.exists())
except NotImplementedError as e:
    print("감성 분류 TODO 미구현:", e)


epoch 1 train loss=0.6857, accuracy=0.5974 | val loss=0.6315, accuracy=0.6570
epoch 2 train loss=0.5624, accuracy=0.7124 | val loss=0.6415, accuracy=0.6700
epoch 3 train loss=0.4678, accuracy=0.7804 | val loss=0.6929, accuracy=0.6640
test loss=0.6355, accuracy=0.6740
sentiment results saved: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/results/sentiment_light.json True


## 9. 저장된 결과 확인

학습/평가 셀을 실행한 뒤 아래 셀에서 저장된 JSON 결과를 바로 확인합니다.


In [18]:
import json

for path in [PRETRAIN_RESULTS_PATH, SENTIMENT_RESULTS_PATH]:
    print("=", path)
    if not path.exists():
        print("아직 파일이 없습니다. 해당 학습/평가 저장 셀을 먼저 실행하세요.")
        continue

    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if payload.get("task") == "pretraining":
        print("train_losses:", payload.get("train_losses"))
        for row in payload.get("eval_history", []):
            print(
                "step", row["global_step"],
                "train_loss", row["train_loss"],
                "val_loss", row["val_loss"],
            )
            print("sample:", row["generated_sample"][:200])
    elif payload.get("task") == "sentiment_classification":
        if payload.get("epoch_history"):
            for row in payload["epoch_history"]:
                print(
                    "epoch", row["epoch"],
                    "train_loss", row.get("train_loss"),
                    "train_accuracy", row.get("train_accuracy"),
                    "val_loss", row.get("val_loss"),
                    "val_accuracy", row.get("val_accuracy"),
                    "test_loss", row.get("test_loss"),
                    "test_accuracy", row.get("test_accuracy"),
                )
        else:
            for row in payload.get("history", []):
                print(
                    row["split"],
                    "epoch", row["epoch"],
                    "loss", row["loss"],
                    "accuracy", row["accuracy"],
                    "num_examples", row["num_examples"],
                )
    print()


= /Users/juhoseok/Desktop/week14-team-03-gpt-lab/results/pretrain_light.json
아직 파일이 없습니다. 해당 학습/평가 저장 셀을 먼저 실행하세요.
= /Users/juhoseok/Desktop/week14-team-03-gpt-lab/results/sentiment_light.json
epoch 3 train_loss 0.46779923050403593 train_accuracy 0.7804 val_loss 0.692896952867508 val_accuracy 0.664 test_loss 0.6355003498792648 test_accuracy 0.674



## 10. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.


In [19]:
run_pytest("tests/")

실행 명령: /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python -m pytest tests/ -v
============================= test session starts ==============================
platform darwin -- Python 3.14.3, pytest-9.0.3, pluggy-1.6.0 -- /Users/juhoseok/Desktop/week14-team-03-gpt-lab/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/juhoseok/Desktop/week14-team-03-gpt-lab
collecting ... collected 34 items

tests/test_attention.py::TestMultiHeadAttention::test_mha_output_shape PASSED [  2%]
tests/test_attention.py::TestMultiHeadAttention::test_mha_causal_mask_future_zero PASSED [  5%]
tests/test_bpe.py::TestSpecialTokens::test_special_ids_fixed PASSED      [  8%]
tests/test_bpe.py::TestBPETokenizer::test_init_special_tokens PASSED     [ 11%]
tests/test_bpe.py::TestBPETokenizer::test_save_load_restores_vocab PASSED [ 14%]
tests/test_bpe.py::TestBPETokenizer::test_encode_decode_restores_original_text PASSED [ 17%]
tests/test_bpe.py::TestBPETokenizer::test_decode_replace_handles_inval

0